<a href="https://colab.research.google.com/github/aamitsharma2705/SSG/blob/main/CLIP_ViT_L_14_336.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate flash-attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 96.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
!pip install -q transformers datasets accelerate

In [2]:
pip install -q transformers torch pillow

**Mount Drive**

In [4]:
from transformers import CLIPModel

model = CLIPModel.from_pretrained(
    "openai/clip-vit-large-patch14-336"
)

print("SUCCESS")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.76k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

SUCCESS
Parameters: 427,944,193


Step 1 — Inspect the Annotation Files

In [5]:
from google.colab import drive
from pathlib import Path

try:
  drive.flush_and_unmount()
except:
  pass

drive.mount('/content/drive')
print(f'drive set up complete!!!')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
drive set up complete!!!


In [7]:
import os
import pickle
from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define the path to your dataset
# Based on your folder structure, the correct path is:
dataset_path = Path('/content/drive/MyDrive/project/SSG/data/ssg_dataset.pkl')

# 3. Read the pickle file
if dataset_path.exists():
    print(f"\nSuccessfully located file at: {dataset_path}")
    print("Loading dataset (this might take a moment depending on file size)...")

    with open(dataset_path, 'rb') as f:
        ssg = pickle.load(f)

    print(type(ssg))
    print(len(ssg))

    first_key = next(iter(ssg))
    print(first_key)
    print(ssg[first_key])

else:
    print(f"\n[ERROR] File not found at: {dataset_path}")
    print("Please double-check that your folder path exactly matches 'project/SSG/data' in your Drive.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Successfully located file at: /content/drive/MyDrive/project/SSG/data/ssg_dataset.pkl
Loading dataset (this might take a moment depending on file size)...
<class 'dict'>
25541
50N4E.mp4/000701.png
[{'class': 'light', 'bbox': None, 'attention_relationship': None, 'spatial_relationship': None, 'contacting_relationship': None, 'metadata': {'tag': '50N4E.mp4/light/000701', 'set': 'train'}, 'visible': False, 'situation': True, 'attributes': {'color': 'white', 'source': 'basement', 'affordance': 'unsure', 'location': 'bathroom', 'state': 'on', 'purpose': 'non_decorative'}, 'contacting_relationship_semantic_role_frame': [{'contacting_relationship': None, 'frame': None}]}, {'class': 'dish', 'bbox': [24.201217747501087, 126.30262249827456, 25.517015054768596, 12.073582425550045], 'attention_relationship': ['looking_at'], 'spatial_relationship

helperfunctions

In [8]:
AGE_MAP = {
    "young_age": "young",
    "middle_age": "middle-aged",
    "old_age": "elderly"
}

SKIP_VALUES = {
    None,
    "",
    "unsure"
}


def clean_text(value):
    if value is None:
        return None

    value = str(value)

    if value.lower() == "unsure":
        return None

    value = value.replace("_", " ")
    value = value.replace("/", " or ")

    return value


def valid_value(value):
    if value is None:
        return False

    if str(value).lower() == "unsure":
        return False

    return True

In [9]:
dataset_path_ag = Path('/content/drive/MyDrive/project/SSG/data/ag_ssg_combined_person.pkl')
with open(dataset_path_ag, "rb") as f:
    person = pickle.load(f)

print(type(person))
print(len(person))

first_key = next(iter(person))
print(first_key)
print(person[first_key])

<class 'dict'>
288782
001YG.mp4/000089.png
{'bbox': array([[ 24.29774 ,  71.443954, 259.23602 , 268.20288 ]], dtype=float32), 'bbox_score': array([0.9960979], dtype=float32), 'bbox_size': (480, 270), 'bbox_mode': 'xyxy', 'keypoints': array([[[149.51952 , 120.54931 ,   1.      ],
        [146.48587 , 111.43697 ,   1.      ],
        [141.09274 , 115.824394,   1.      ],
        [111.76759 , 123.58676 ,   1.      ],
        [112.44173 , 124.26174 ,   1.      ],
        [ 82.10537 , 154.6362  ,   1.      ],
        [113.45295 , 168.47343 ,   1.      ],
        [153.56436 , 207.96022 ,   1.      ],
        [162.66527 , 247.44699 ,   1.      ],
        [146.48587 , 149.91127 ,   1.      ],
        [216.59659 , 229.22232 ,   1.      ],
        [112.10466 , 243.73456 ,   1.      ],
        [163.3394  , 267.69662 ,   1.      ],
        [237.83205 , 202.56032 ,   1.      ],
        [239.18031 , 202.56032 ,   1.      ],
        [186.93436 , 219.0975  ,   1.      ],
        [220.9785  , 227.87234

Step 2 — Verify Frame Coverage

In [10]:
from pathlib import Path

FRAME_ROOT = Path("/content/drive/MyDrive/project/SSG/data/frames")

sample_keys = list(ssg.keys())[:100]

for key in sample_keys:
    jpg = FRAME_ROOT / key.replace(".png", ".jpg")
    print(jpg.exists(), jpg)

True /content/drive/MyDrive/project/SSG/data/frames/50N4E.mp4/000701.jpg
True /content/drive/MyDrive/project/SSG/data/frames/CUZND.mp4/000473.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8D464.mp4/000466.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8D464.mp4/000524.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8AZRX.mp4/000838.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8AZRX.mp4/000859.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8AZRX.mp4/000886.jpg
True /content/drive/MyDrive/project/SSG/data/frames/CO3LU.mp4/000944.jpg
True /content/drive/MyDrive/project/SSG/data/frames/CO3LU.mp4/000960.jpg
True /content/drive/MyDrive/project/SSG/data/frames/6AJX0.mp4/000246.jpg
True /content/drive/MyDrive/project/SSG/data/frames/6AJX0.mp4/000285.jpg
True /content/drive/MyDrive/project/SSG/data/frames/5BQMX.mp4/000091.jpg
True /content/drive/MyDrive/project/SSG/data/frames/A84J7.mp4/000522.jpg
True /content/drive/MyDrive/project/SSG/data/frames

Caption Generator

In [ ]:
def build_object_sentence(obj_class, attrs):

    color = clean_text(attrs.get("color"))
    material = clean_text(attrs.get("material"))
    location = clean_text(attrs.get("location"))
    state = clean_text(attrs.get("state"))
    affordance = clean_text(attrs.get("affordance"))

    # Base description
    description_parts = []

    if color:
        description_parts.append(color)

    if material:
        description_parts.append(material)

    description_parts.append(obj_class.replace("/", " or "))

    description = " ".join(description_parts)

    # Action-oriented description
    if state == "held" or location == "hand":
        return f"A {description} is being held in the hand."

    if state:
        return f"A {description} is {state}."

    if location:
        return f"A {description} is located in the {location}."

    return f"A {description} is present."

In [ ]:
def build_caption(frame_key, ssg, person_data):
    sentences = []

    # --------------------
    # Person description
    # --------------------
    if frame_key in person_data:
        person_entry = person_data[frame_key]

        # If it's a list or unexpected type, raise an error so the loop logs it
        if isinstance(person_entry, list):
            raise ValueError(f"Expected dict under frame_key, but got list: {type(person_entry)}")
        elif not isinstance(person_entry, dict):
            raise ValueError(f"Unexpected data type under frame_key: {type(person_entry)}")

        attrs = person_entry.get("attributes", {})
        if not isinstance(attrs, dict):
            raise ValueError(f"Expected dict for 'attributes', but got: {type(attrs)}")

        # Safely fetch and clean attributes
        raw_age = attrs.get("age")
        age = AGE_MAP.get(raw_age, clean_text(raw_age))

        gender = clean_text(attrs.get("gender"))
        dress = clean_text(attrs.get("dress_color"))
        hair = clean_text(attrs.get("hair_color"))
        skin = clean_text(attrs.get("skin_color"))

        # Build the sentence pieces conditionally
        person_parts = []
        if age:
            person_parts.append(f"A {age.replace('_', ' ')}")
        else:
            person_parts.append("A")

        if gender:
            person_parts.append(f"{gender} person")
        else:
            person_parts.append("person")

        details = []
        if skin:
            details.append(f"{skin} skin")
        if hair:
            details.append(f"{hair} hair")
        if dress:
            details.append(f"{dress} clothing")

        if details:
            if len(details) > 1:
                details_str = ", ".join(details[:-1]) + f" and {details[-1]}"
            else:
                details_str = details[0]
            person_sentence = " ".join(person_parts) + f" with {details_str}."
        else:
            person_sentence = " ".join(person_parts) + "."

        sentences.append(person_sentence)

    # --------------------
    # Objects
    # --------------------
    for obj in ssg[frame_key]:
        obj_class = obj["class"]
        attrs = obj.get("attributes", {})

        attr_parts = []
        for k, v in attrs.items():
            if not valid_value(v):
                continue
            attr_parts.append(f"{k} {clean_text(v)}")

        if attr_parts:
            obj_sentence = build_object_sentence(obj_class, attrs)
        else:
            obj_sentence = f"There is a {obj_class}."
        sentences.append(obj_sentence)

        # --------------------
        # Relationship SRV
        # --------------------
        frames = obj.get("contacting_relationship_semantic_role_frame", [])
        for rel in frames:
            relation = rel.get("contacting_relationship")
            frame = rel.get("frame")
            if not relation or not frame:
                continue

            place = clean_text(frame.get("place"))
            body = clean_text(frame.get("body_part"))
            entity = clean_text(frame.get("entity"))

            if not place or not body or not entity:
                continue

            rel_sentence = (
                f"The person is {relation.replace('_', ' ')} "
                f"the {entity} using the {body} in the {place}."
            )
            sentences.append(rel_sentence)

    return " ".join(sentences)

Generate JSONL

In [ ]:
import json
from pathlib import Path

records = []
corrupt_or_missing_frames = []  # <--- List to collect your bad entries

total = len(ssg)
processed = 0

for frame_key in ssg.keys():
    img_path = FRAME_ROOT / frame_key.replace(".png", ".jpg")

    # Track missing image frames as well if needed
    if not img_path.exists():
        corrupt_or_missing_frames.append({"frame_key": frame_key, "reason": "Missing JPEG file"})
        continue

    try:
        caption = build_caption(frame_key, ssg, person)
        records.append({
            "image": str(img_path),
            "text": caption
        })
    except (AttributeError, ValueError) as e:
        # Capture data type issues / structure mismatches dynamically
        print(f"\n[SKIPPED] Found bad entry at key: {frame_key} | Error: {e}")
        corrupt_or_missing_frames.append({"frame_key": frame_key, "reason": str(e)})
        continue

    processed += 1
    if processed % 100 == 0:
        print(f"Processed {processed :,} / {total :,} ({processed/total:.1%})")

# Write valid records out to dataset
with open("train.jsonl", "w") as f:
    for row in records:
        f.write(json.dumps(row) + "\n")

print(f"\nCompleted: {processed :,} records successfully written.")
print(f"Total invalid entries mapped for filtering: {len(corrupt_or_missing_frames)}")

Processed 100 / 25,541 (0.4%)
Processed 200 / 25,541 (0.8%)
Processed 300 / 25,541 (1.2%)
Processed 400 / 25,541 (1.6%)
Processed 500 / 25,541 (2.0%)
Processed 600 / 25,541 (2.3%)
Processed 700 / 25,541 (2.7%)
Processed 800 / 25,541 (3.1%)
Processed 900 / 25,541 (3.5%)
Processed 1,000 / 25,541 (3.9%)
Processed 1,100 / 25,541 (4.3%)
Processed 1,200 / 25,541 (4.7%)
Processed 1,300 / 25,541 (5.1%)
Processed 1,400 / 25,541 (5.5%)
Processed 1,500 / 25,541 (5.9%)
Processed 1,600 / 25,541 (6.3%)
Processed 1,700 / 25,541 (6.7%)
Processed 1,800 / 25,541 (7.0%)
Processed 1,900 / 25,541 (7.4%)
Processed 2,000 / 25,541 (7.8%)
Processed 2,100 / 25,541 (8.2%)
Processed 2,200 / 25,541 (8.6%)
Processed 2,300 / 25,541 (9.0%)
Processed 2,400 / 25,541 (9.4%)
Processed 2,500 / 25,541 (9.8%)
Processed 2,600 / 25,541 (10.2%)
Processed 2,700 / 25,541 (10.6%)
Processed 2,800 / 25,541 (11.0%)
Processed 2,900 / 25,541 (11.4%)
Processed 3,000 / 25,541 (11.7%)
Processed 3,100 / 25,541 (12.1%)
Processed 3,200 / 25

In [ ]:
# Print the first 10 bad frames to see what went wrong
import pandas as pd
df_bad = pd.DataFrame(corrupt_or_missing_frames)
print(df_bad.head(10))

# Optional: Save it to a text/CSV file so you can load it to clean your pkl later
df_bad.to_csv("bad_frames_log.csv", index=False)

              frame_key                                             reason
0  1D2OX.mp4/000624.png                                  Missing JPEG file
1  1D2OX.mp4/000614.png                                  Missing JPEG file
2  1D2OX.mp4/000789.png                                  Missing JPEG file
3  1D2OX.mp4/000735.png                                  Missing JPEG file
4  1D2OX.mp4/000802.png                                  Missing JPEG file
5  004QE.mp4/000217.png  Expected dict for 'attributes', but got: <clas...
6  004QE.mp4/000276.png  Expected dict for 'attributes', but got: <clas...
7  004QE.mp4/000312.png  Expected dict for 'attributes', but got: <clas...
8  004QE.mp4/000264.png  Expected dict for 'attributes', but got: <clas...
9  00NN7.mp4/000820.png  Expected dict for 'attributes', but got: <clas...


Split the dataset.

In [ ]:
from sklearn.model_selection import train_test_split

train_records, val_records = train_test_split(
    records,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print(len(train_records))
print(len(val_records))

22977
2554


Save JSONL - train and val


In [ ]:
import json

with open("train.jsonl", "w") as f:
    for item in train_records:
        f.write(json.dumps(item) + "\n")

with open("val.jsonl", "w") as f:
    for item in val_records:
        f.write(json.dumps(item) + "\n")

verify


In [ ]:
!head -n 2 train.jsonl

{"image": "/content/drive/MyDrive/project/SSG/data/frames/136V6.mp4/001010.jpg", "text": "A middle-aged male person with dark skin, black hair and multiple clothing. A brown broom is being held in the hand. The person is holding the broom using the hand in the bedroom."}
{"image": "/content/drive/MyDrive/project/SSG/data/frames/CM5SK.mp4/000016.jpg", "text": "A middle-aged male person with medium skin, bald hair and beige clothing. A white refrigerator is closed. The person is holding the refrigerator using the hand in the kitchen."}


In [ ]:
print(len(train_records))
print(len(val_records))

22977
2554


In [ ]:
import json
import random
from PIL import Image

with open("train.jsonl") as f:
    lines = f.readlines()

for line in random.sample(lines, 100):
    sample = json.loads(line)

    img = Image.open(sample["image"])
    img.verify()

print("All sampled images valid")

All sampled images valid


CLIp install

In [11]:
!git clone https://github.com/mlfoundations/open_clip.git


Cloning into 'open_clip'...
remote: Enumerating objects: 4861, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 4861 (delta 175), reused 155 (delta 151), pack-reused 4604 (from 2)
Receiving objects: 100% (4861/4861), 16.10 MiB | 21.03 MiB/s, done.
Resolving deltas: 100% (3016/3016), done.


In [14]:
!pip show open_clip_torch

Name: open_clip_torch
Version: 4.2.0.dev0
Summary: Open reproduction of consastive language-image pretraining (CLIP) and related.
Home-page: https://github.com/mlfoundations/open_clip
Author: Gabriel Ilharco, Mitchell Wortsman, Romain Beaumont
Author-email: Ross Wightman <ross@huggingface.co>
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Editable project location: /content/open_clip
Requires: ftfy, huggingface-hub, regex, safetensors, timm, torch, torchvision, tqdm
Required-by: 


In [13]:
%cd /content/open_clip

!pip install -e .

/content/open_clip
Obtaining file:///content/open_clip
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
  Building editable for open_clip_torch (pyproject.toml) ... done
  Created wheel for open_clip_torch: filename=open_clip_torch-4.2.0.dev0-py3-none-any.whl size=17836 sha256=dfdf3cbe7b4503d57f3a133b36448b29dcbb86e07a013936b59d95ac2fb02b20
  Stored in directory: /tmp/pip-ephem-wheel-cache-tmtcppbp/wheels/73/44/bc/cb189915994f882b7472b4af15249b65016f6adde1db4e0bdc
Successfully built open_clip_torch


In [15]:
import sys
import os

# 1. Ensure we are inside the open_clip folder or it's in your system path
if os.path.exists('/content/open_clip/src'):
    sys.path.append('/content/open_clip/src')

# 2. Import the correct package from the source folder
import open_clip

# 3. Test the methods
print("Available models:", open_clip.list_models()[:5])

# Load your model
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-L-14',
    pretrained='openai'
)
print("\nModel loaded successfully!")

Available models: ['CLAP-HTSAT-base', 'CLAP-HTSAT-base-Roberta-base', 'CLAP-HTSAT-large', 'CLAP-HTSAT-tiny', 'CLAP-HTSAT-tiny-Roberta-base']


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/content/open_clip/src/open_clip/factory.py:476: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(



Model loaded successfully!


In [16]:
import open_clip

models = open_clip.list_models()
[m for m in models if "ViT-L-14" in m]

['coca_ViT-L-14',
 'ViT-L-14',
 'ViT-L-14-280',
 'ViT-L-14-336',
 'ViT-L-14-336-quickgelu',
 'ViT-L-14-CLIPA',
 'ViT-L-14-CLIPA-336',
 'ViT-L-14-quickgelu',
 'ViT-L-14-worldwide',
 'ViT-L-14-worldwide-quickgelu']

In [17]:
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-L-14-336',
    pretrained='openai'
)

print("Loaded 336 model")

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loaded 336 model


In [18]:
!find /content/open_clip -name "main.py"

/content/open_clip/src/open_clip_train/main.py


In [19]:
!find /content/open_clip -type d | grep train

/content/open_clip/src/open_clip_train


Install Training Dependencies

In [20]:
%cd /content/open_clip

!pip install -r requirements-training.txt

/content/open_clip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.2 MB/s eta 0:00:00


Verify Training Module

In [21]:
!python -m open_clip_train.main --help

2026-06-13 12:57:08.307084: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-13 12:57:08.380974: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
usage: main.py [-h] [--train-data TRAIN_DATA]
               [--train-data-upsampling-factors TRAIN_DATA_UPSAMPLING_FACTORS]
               [--val-data VAL_DATA] [--train-num-samples TRAIN_NUM_SAMPLES]
               [--val-num-samples VAL_NUM_SAMPLES]
               [--dataset-type {webdataset,webdataset-audio,csv,synthetic,synthetic-audio,auto}]
               [--audio-

Create Smoke Test Files

In [ ]:
import json

# 1. Change your directory back to the main Colab workspace root
%cd /content

# 2. Double-check that the files are right there
print("Files in workspace root:", os.listdir('.'))
# Load the records back from the saved JSONL files
train_records = []
with open("train.jsonl", "r") as f:
    for line in f:
        train_records.append(json.loads(line))

val_records = []
with open("val.jsonl", "r") as f:
    for line in f:
        val_records.append(json.loads(line))

print(f"Successfully reloaded {len(train_records)} train records.")
print(f"Successfully reloaded {len(val_records)} val records.")

# Now create your smoke test subsets safely
small_train = train_records[:1000]
small_val = val_records[:100]

with open("train_small.jsonl", "w") as f:
    for r in small_train:
        f.write(json.dumps(r) + "\n")

with open("val_small.jsonl", "w") as f:
    for r in small_val:
        f.write(json.dumps(r) + "\n")

print("\nSmoke test files 'train_small.jsonl' and 'val_small.jsonl' created successfully!")

/content
Files in workspace root: ['.config', 'drive', 'train.jsonl', 'open_clip', '.ipynb_checkpoints', 'bad_frames_log.csv', 'val.jsonl', 'sample_data']
Successfully reloaded 22977 train records.
Successfully reloaded 2554 val records.

Smoke test files 'train_small.jsonl' and 'val_small.jsonl' created successfully!


Verify Dataset Format

In [ ]:
import json

with open("train_small.jsonl") as f:
    sample = json.loads(next(f))

print(sample)

{'image': '/content/drive/MyDrive/project/SSG/data/frames/136V6.mp4/001010.jpg', 'text': 'A middle-aged male person with dark skin, black hair and multiple clothing. A brown broom is being held in the hand. The person is holding the broom using the hand in the bedroom.'}


Smoke Test Training

jsonl not working converting to csv

In [ ]:
import json
import pandas as pd

rows = []

with open("train.jsonl") as f:
    for line in f:
        rec = json.loads(line)

        rows.append({
            "filepath": rec["image"],
            "caption": rec["text"]
        })

train_df = pd.DataFrame(rows)

train_df.to_csv(
    "train.csv",
    index=False
)

print(len(train_df))

22977


In [ ]:
rows = []

with open("val.jsonl") as f:
    for line in f:
        rec = json.loads(line)

        rows.append({
            "filepath": rec["image"],
            "caption": rec["text"]
        })

val_df = pd.DataFrame(rows)

val_df.to_csv(
    "val.csv",
    index=False
)

print(len(val_df))

2554


verify command

In [22]:
!python -m open_clip_train.main --help | grep csv

2026-06-13 12:57:57.660926: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-13 12:57:57.732540: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
               [--dataset-type {webdataset,webdataset-audio,csv,synthetic,synthetic-audio,auto}]
               [--dataset-resampled] [--csv-separator CSV_SEPARATOR]
               [--csv-img-key CSV_IMG_KEY] [--csv-caption-key CSV_CAPTION_KEY]
  --dataset-type {webdataset,webdataset-audio,csv,synthetic,synthetic-audio,auto}
  --csv-separator CSV_SEPARATOR
               

In [ ]:
!python -m open_clip_train.main \
  --train-data train_small.csv \
  --val-data val_small.csv \
  --dataset-type csv \
  --csv-separator "," \
  --csv-img-key filepath \
  --csv-caption-key caption \
  --model ViT-L-14-336 \
  --pretrained openai \
  --batch-size 2 \
  --accum-freq 8 \
  --lr 5e-6 \
  --epochs 1 \
  --workers 2 \
  --precision amp \
  --logs logs_smoke

2026-06-13 12:06:42.943243: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-13,12:06:56 | INFO | open_clip_train.main | Running with a single process. Device cuda.
2026-06-13,12:06:56 | INFO | open_clip.factory | Parsing model identifier. Schema: None, Identifier: ViT-L-14-336
2026-06-13,12:06:56 | INFO | open_clip.factory | Loaded built-in ViT-L-14-336 model config.
2026-06-13,12:06:56 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/timm/vit_large_patch14_clip_336.openai/resolve/main/open_clip_model.safetensors "HTTP/1.1 302 Found"
2026-06-13,12:06:56 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads

In [ ]:
!find logs_smoke -name "*.pt"

logs_smoke/2026_06_13-12_06_56-model_ViT-L-14-336-lr_5e-06-b_2-j_2-p_amp/checkpoints/epoch_1.pt


# **Full training**

Path changed to /content/open_clip hence mentione /content/train or cd  back to main directory

In [24]:
!python -m open_clip_train.main \
  --train-data /content/train.csv \
  --val-data /content/val.csv \
  --dataset-type csv \
  --csv-separator "," \
  --csv-img-key filepath \
  --csv-caption-key caption \
  --model ViT-L-14-336 \
  --pretrained openai \
  --batch-size 2 \
  --accum-freq 8 \
  --lr 5e-6 \
  --epochs 3 \
  --workers 2 \
  --precision amp \
  --logs logs_ssg

2026-06-13 13:03:25.897567: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-13 13:03:25.969658: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-13,13:03:34 | INFO | open_clip_train.main | Running with a single process. Device cuda.
2026-06-13,13:03:34 | INFO | open_clip.factory | Parsing model identifier. Schema: None, Identifier: ViT-L-14-336
2026-06-13,13:03:34 | INFO | open_clip.factory | Loaded built-in ViT-L-14-336 model config.
2026-06-13,13:03:34 | INFO | httpx | HTTP Request: HEAD https://huggingf

back up files

In [25]:
# 1. Create a dedicated backup folder in your Google Drive
!mkdir -p "/content/drive/MyDrive/project/SSG/data/open_clip_backups"

# 2. Copy your completed training run logs and checkpoints to your Drive
!cp -r /content/open_clip/logs_ssg/2026_06_13-13_03_34-model_ViT-L-14-336-lr_5e-06-b_2-j_2-p_amp "/content/drive/MyDrive/project/SSG/data/open_clip_backups/"

print("✅ All checkpoints (including epoch_3.pt) successfully backed up to your permanent Google Drive!")

✅ All checkpoints (including epoch_3.pt) successfully backed up to your permanent Google Drive!


verify check point

In [26]:
import os

drive_path = "/content/drive/MyDrive/project/SSG/data/open_clip_backups/2026_06_13-13_03_34-model_ViT-L-14-336-lr_5e-06-b_2-j_2-p_amp/checkpoints"

if os.path.exists(drive_path):
    print("🎉 Verification Complete! Your trained parameters are locked in your cloud:")
    print(os.listdir(drive_path))
else:
    print("❌ Backup folder not found on Drive. Please double check that Step 1 completed successfully.")

🎉 Verification Complete! Your trained parameters are locked in your cloud:
['results.jsonl', 'epoch_1.pt', 'epoch_2.pt', 'epoch_3.pt']


# **Validation for Clip**

Load Validation data

In [29]:
import pandas as pd

# 1. Change your directory back to the main Colab workspace root
%cd /content

val_df = pd.read_csv("val.csv")

print(len(val_df))
val_df.head()

/content
2554


,filepath,caption
0,/content/drive/MyDrive/project/SSG/data/frames...,"A middle-aged male person with dark skin, blac..."
1,/content/drive/MyDrive/project/SSG/data/frames...,"A young male person with fair skin, black hair..."
2,/content/drive/MyDrive/project/SSG/data/frames...,"A young female person with fair skin, brown ha..."
3,/content/drive/MyDrive/project/SSG/data/frames...,"A young female person with fair skin, blonde h..."
4,/content/drive/MyDrive/project/SSG/data/frames...,"A child male person with dark skin, black hair..."


helper functions

In [30]:
import torch
import open_clip
from PIL import Image
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"


def encode_dataset(model, preprocess, tokenizer, df):
    image_features = []
    text_features = []

    model.eval()

    with torch.no_grad():
        for _, row in tqdm(df.iterrows(), total=len(df)):

            image = preprocess(
                Image.open(row["filepath"]).convert("RGB")
            ).unsqueeze(0).to(device)

            text = tokenizer(
                [row["caption"]]
            ).to(device)

            img_feat = model.encode_image(image)
            txt_feat = model.encode_text(text)

            img_feat /= img_feat.norm(dim=-1, keepdim=True)
            txt_feat /= txt_feat.norm(dim=-1, keepdim=True)

            image_features.append(
                img_feat.cpu()
            )

            text_features.append(
                txt_feat.cpu()
            )

    image_features = torch.cat(image_features)
    text_features = torch.cat(text_features)

    return image_features, text_features

Retrieval metrics

In [31]:
def retrieval_metrics(
    image_features,
    text_features,
    ks=(1, 5, 10)
):

    sims = image_features @ text_features.T

    n = sims.shape[0]

    results = {}

    # image -> text

    ranks = []

    for i in range(n):
        ranking = torch.argsort(
            sims[i],
            descending=True
        )

        rank = (
            ranking == i
        ).nonzero(
            as_tuple=True
        )[0].item()

        ranks.append(rank)

    for k in ks:
        results[f"i2t_R@{k}"] = (
            sum(r < k for r in ranks)
            / n
        )

    # text -> image

    ranks = []

    for i in range(n):
        ranking = torch.argsort(
            sims[:, i],
            descending=True
        )

        rank = (
            ranking == i
        ).nonzero(
            as_tuple=True
        )[0].item()

        ranks.append(rank)

    for k in ks:
        results[f"t2i_R@{k}"] = (
            sum(r < k for r in ranks)
            / n
        )

    return results

Evaluate OpenAI CLIP

In [32]:
import pandas as pd

# 1. Load your baseline model
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14-336",
    pretrained="openai"
)
model = model.to(device)
tokenizer = open_clip.get_tokenizer("ViT-L-14-336")

# 2. Explicitly load val.csv right here
val_csv_df = pd.read_csv("val.csv")

# 3. Pass it to the encoder
img_feats, txt_feats = encode_dataset(
    model,
    preprocess,
    tokenizer,
    val_csv_df  # Using the newly loaded DataFrame from val.csv
)

# 4. Compute metrics
baseline_metrics = retrieval_metrics(img_feats, txt_feats)
print("OpenAI Baseline Metrics on val.csv:")
print(baseline_metrics)

/content/open_clip/src/open_clip/factory.py:476: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
100%|██████████| 2554/2554 [18:52<00:00,  2.26it/s]


OpenAI Baseline Metrics on val.csv:
{'i2t_R@1': 0.06225528582615505, 'i2t_R@5': 0.1851996867658575, 'i2t_R@10': 0.2572435395458105, 't2i_R@1': 0.048159749412685984, 't2i_R@5': 0.13469068128425998, 't2i_R@10': 0.19498825371965545}


Evaluate Fine-Tuned CLIP

In [35]:
import torch
import open_clip

# 1. Provide the exact path to your fine-tuned epoch 3 parameters
CHECKPOINT = "/content/open_clip/logs_ssg/2026_06_13-13_03_34-model_ViT-L-14-336-lr_5e-06-b_2-j_2-p_amp/checkpoints/epoch_3.pt"

# 2. Re-create the base model architecture shell (uninitialized)
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14-336",
    pretrained=None
)

# 3. Load your fine-tuned weights into memory safely
print("Loading fine-tuned parameters from checkpoint...")
ckpt = torch.load(CHECKPOINT, map_location="cpu")
model.load_state_dict(ckpt["state_dict"], strict=False)

# 4. Push the initialized model over to your GPU
model = model.to(device)
tokenizer = open_clip.get_tokenizer("ViT-L-14-336")

print("✨ Fine-tuned model loaded successfully on GPU! Encoding validation dataset...")

# 5. Extract features using your fine-tuned weights
img_feats, txt_feats = encode_dataset(
    model,
    preprocess,
    tokenizer,
    val_df
)

# 6. Calculate your updated performance metrics
ft_metrics = retrieval_metrics(img_feats, txt_feats)
print("\nFine-Tuned Metrics on val.csv:")
print(ft_metrics)

Loading fine-tuned parameters from checkpoint...
✨ Fine-tuned model loaded successfully on GPU! Encoding validation dataset...


100%|██████████| 2554/2554 [02:54<00:00, 14.64it/s]



Fine-Tuned Metrics on val.csv:
{'i2t_R@1': 0.3128425998433829, 'i2t_R@5': 0.6805011746280345, 'i2t_R@10': 0.7948316366483946, 't2i_R@1': 0.3042286609240407, 't2i_R@5': 0.6832419733750978, 't2i_R@10': 0.7956147220046985}


# Comparison

| Metric   | OpenAI CLIP | Fine-Tuned SSG CLIP | Improvement |
| -------- | ----------: | ------------------: | ----------: |
| i2t R@1  |       6.23% |              31.28% |  **+25.1%** |
| i2t R@5  |      18.52% |              68.05% |  **+49.5%** |
| i2t R@10 |      25.72% |              79.48% |  **+53.8%** |
| t2i R@1  |       4.82% |              30.42% |  **+25.6%** |
| t2i R@5  |      13.47% |              68.32% |  **+54.9%** |
| t2i R@10 |      19.50% |              79.56% |  **+60.1%** |
